# 03 — Seleção de K (orientada ao negócio) + KMeans

Objetivos:
- Avaliar candidatos de K com métricas técnicas e critérios de negócio
- Medir estabilidade (ARI/NMI) via subamostragem
- Escolher K por regra explícita e re-treinar o pipeline final
- Salvar o pipeline em `reports/kmeans_pipeline.joblib`


In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve()))

import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.metrics import adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score, normalized_mutual_info_score, silhouette_score

from src.features import build_modeling_dataframe
from src.modeling import build_kmeans_pipeline, save_pipeline, transform_for_model

sns.set_theme(style='whitegrid', palette='Set2')

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
ASSETS_DIR = REPORTS_DIR / 'assets'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
df_cliente = pd.read_parquet(PROCESSED_DIR / 'base_cliente_atual.parquet')
X, spec = build_modeling_dataframe(df_cliente)
X.shape, spec

## 1) Candidatos de K (obrigatórios)

Avaliamos K em {3,4,5,6,7,8,9,10,12} e registramos:
- Inertia (Elbow)
- Silhouette, Calinski-Harabasz, Davies-Bouldin (em amostra reprodutível)
- Tamanho mínimo do cluster (% e contagem)


In [ ]:
candidates = [3, 4, 5, 6, 7, 8, 9, 10, 12]
rng = np.random.default_rng(42)
sample_n = min(6000, len(X))
sample_idx = rng.choice(len(X), size=sample_n, replace=False)

rows = []
sizes_by_k = {}

for k in candidates:
    pipe = build_kmeans_pipeline(spec, k=k, random_state=42)
    pipe.fit(X)
    inertia = float(pipe.named_steps['kmeans'].inertia_)
    
    Xs = X.iloc[sample_idx].copy()
    Xt = transform_for_model(pipe, Xs)
    labels_s = pipe.predict(Xs)
    Xt_dense = Xt.toarray() if hasattr(Xt, 'toarray') else np.asarray(Xt)
    
    sil = float(silhouette_score(Xt_dense, labels_s))
    ch = float(calinski_harabasz_score(Xt_dense, labels_s))
    db = float(davies_bouldin_score(Xt_dense, labels_s))
    
    labels_full = pipe.predict(X)
    counts = pd.Series(labels_full).value_counts().sort_index()
    min_n = int(counts.min())
    min_pct = float(min_n / len(X))
    
    sizes_by_k[k] = counts.rename('n').to_frame().reset_index().rename(columns={'index':'cluster_id'})
    
    rows.append({
        'k': k,
        'inertia': inertia,
        'silhouette': sil,
        'calinski_harabasz': ch,
        'davies_bouldin': db,
        'min_cluster_n': min_n,
        'min_cluster_pct': min_pct,
    })

k_table = pd.DataFrame(rows).sort_values('k').reset_index(drop=True)
k_table

## 2) Estabilidade (ARI/NMI) por subamostragem

Treinamos 5 vezes com 70% da base (seeds controladas) e medimos concordância das rotulações na mesma amostra de avaliação.

In [ ]:
eval_n = min(5000, len(X))
eval_idx = np.random.default_rng(42).choice(len(X), size=eval_n, replace=False)

stab_rows = []
for k in candidates:
    labels_runs = []
    for i in range(5):
        seed = 42 + i
        rng_i = np.random.default_rng(seed)
        train_idx = rng_i.choice(len(X), size=int(len(X) * 0.7), replace=False)
        pipe = build_kmeans_pipeline(spec, k=k, random_state=42)
        pipe.fit(X.iloc[train_idx])
        labels_runs.append(pipe.predict(X.iloc[eval_idx]))
    for (i, a), (j, b) in itertools.combinations(list(enumerate(labels_runs)), 2):
        stab_rows.append({
            'k': k,
            'pair': f'{i}-{j}',
            'ari': float(adjusted_rand_score(a, b)),
            'nmi': float(normalized_mutual_info_score(a, b)),
        })

stability = pd.DataFrame(stab_rows)
stability.groupby('k')[['ari','nmi']].median().reset_index()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=stability, x='k', y='ari', ax=ax[0])
ax[0].set_title('Estabilidade por K (ARI)')
ax[0].set_xlabel('K')
ax[0].set_ylabel('ARI')

sns.boxplot(data=stability, x='k', y='nmi', ax=ax[1])
ax[1].set_title('Estabilidade por K (NMI)')
ax[1].set_xlabel('K')
ax[1].set_ylabel('NMI')

fig.tight_layout()
fig.savefig(ASSETS_DIR / 'k_stability.png', dpi=160, bbox_inches='tight')
fig

## 3) Regra explícita de decisão (orientada ao negócio)

Regra aplicada:
- Eliminar K em que algum cluster <2% (exceto achado crítico justificável)
- Preferir região do “joelho” do Elbow desde que Silhouette e estabilidade não piorem muito
- Desempatar por acionabilidade (clareza de perfis e presença de grupos como “pagantes sem uso”)


In [ ]:
stab_med = stability.groupby('k')[['ari','nmi']].median().reset_index().rename(columns={'ari':'ari_med','nmi':'nmi_med'})
kt = k_table.merge(stab_med, on='k', how='left')
kt['elegivel'] = kt['min_cluster_pct'] >= 0.02

cand = kt[kt['elegivel']].copy()
if cand.empty:
    cand = kt.copy()

score = 0
score += cand['silhouette'].rank(pct=True)
score += cand['ari_med'].rank(pct=True)
score += cand['calinski_harabasz'].rank(pct=True)
score -= cand['davies_bouldin'].rank(pct=True)
score += cand['min_cluster_pct'].rank(pct=True)
score -= cand['k'].rank(pct=True) * 0.15
cand['score_selecao'] = score

k_escolhido = int(cand.sort_values(['score_selecao','silhouette'], ascending=False).iloc[0]['k'])
k_escolhido, cand.sort_values('k')

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
sns.lineplot(data=k_table, x='k', y='inertia', marker='o', ax=ax[0,0])
ax[0,0].axvline(k_escolhido, color='black', linestyle='--', linewidth=1)
ax[0,0].set_title('Elbow (Inertia)')

sns.lineplot(data=k_table, x='k', y='silhouette', marker='o', ax=ax[0,1])
ax[0,1].axvline(k_escolhido, color='black', linestyle='--', linewidth=1)
ax[0,1].set_title('Silhouette (amostra)')

sns.lineplot(data=k_table, x='k', y='calinski_harabasz', marker='o', ax=ax[1,0])
ax[1,0].axvline(k_escolhido, color='black', linestyle='--', linewidth=1)
ax[1,0].set_title('Calinski-Harabasz (amostra)')

sns.lineplot(data=k_table, x='k', y='davies_bouldin', marker='o', ax=ax[1,1])
ax[1,1].axvline(k_escolhido, color='black', linestyle='--', linewidth=1)
ax[1,1].set_title('Davies-Bouldin (amostra; menor é melhor)')

fig.suptitle(f'Seleção de K — K escolhido: {k_escolhido}', y=1.02)
fig.tight_layout()
fig.savefig(ASSETS_DIR / 'k_selection.png', dpi=160, bbox_inches='tight')
fig

## 4) Treino final e salvamento do pipeline

In [ ]:
pipe_final = build_kmeans_pipeline(spec, k=k_escolhido, random_state=42)
pipe_final.fit(X)
save_pipeline(pipe_final, str(REPORTS_DIR / 'kmeans_pipeline.joblib'))
pipe_final